# Benchmark de Quantum Simulator en AMD ROCm

Este notebook ejecuta el benchmark del simulador statevector en la GPU de AMD (W7900D, RDNA 3, gfx1100) detectando el hardware en runtime y generando archivos de salida.

**Flujo verificado:**
1. Clona el repositorio del hackathon desde GitHub.
2. **Descubre dinámicamente** dónde están `qsim_torch.py`, `hardware_wheels.py`, `benchmark_notebook.py` sin asumir rutas fijas ni nombres de subdirectorios.
3. Instala dependencias.
4. Detecta hardware (GPU, arquitectura GFX, versión ROCm).
5. Ejecuta barrido de benchmarks (n × batch × dtype).
6. Genera `benchmark_results_*.json` y `BENCHMARK_SUMMARY_*.md`.

**IMPORTANTE:** Guarda los archivos generados bajo `/persistent` (o `/workspace` según tu URL de sesión) para que sobrevivan al apagar la sesión.

**No hardcodea rutas de usuario ni nombres de subdirectorios.** Todo se descubre en runtime.

In [ ]:
# ============================================================================
# PASO 1: Obtener el código del repositorio
# ============================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jalp17/chern-calibration-copilot.git"
REPO_NAME = "chern-calibration-copilot"

print(f"Directorio de trabajo actual: {Path.cwd()}")

# Clonar el repositorio en el directorio de trabajo actual
repo_dir = Path.cwd() / REPO_NAME

if repo_dir.exists() and (repo_dir / ".git").exists():
    print(f"\nEl repositorio ya existe en {repo_dir}. Actualizando...")
    result = subprocess.run(
        ["git", "-C", str(repo_dir), "pull"],
        capture_output=True, text=True
    )
    print(result.stdout)
else:
    print(f"\nClonando {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Fallo al clonar el repositorio: {result.stderr}")
    print("✓ Repositorio clonado correctamente")

# ============================================================================
# PASO 1b: Descubrir dónde están los archivos clave
# ============================================================================

ARCHIVOS_CLAVE = [
    "qsim_torch.py",
    "hardware_wheels.py",
    "benchmark_notebook.py",
]

simulador_dir = None

# Buscar recursivamente en el repositorio clonado
for ruta in repo_dir.rglob("qsim_torch.py"):
    simulador_dir = ruta.parent
    print(f"Encontrado qsim_torch.py en: {simulador_dir}")
    break

if simulador_dir is None:
    raise RuntimeError(
        f"No se encontró qsim_torch.py en el repositorio {repo_dir}."
        " Verificar que el repositorio tenga el código del simulador."
    )

# Verificar que los otros archivos clave estén en el mismo directorio
for archivo in ARCHIVOS_CLAVE:
    ruta = simulador_dir / archivo
    if not ruta.exists():
        # Buscar en cualquier parte del repo como fallback
        encontrados = list(repo_dir.rglob(archivo))
        if encontrados:
            simulador_dir = encontrados[0].parent
            print(f"Reubicado {archivo} en: {simulador_dir}")
        else:
            raise RuntimeError(
                f"No se encontró {archivo} en el repositorio."
            )

print(f"\nDirectorio del simulador: {simulador_dir}")
print(f"Archivos encontrados:")
for archivo in ARCHIVOS_CLAVE:
    ruta = simulador_dir / archivo
    print(f"  {archivo}: {'EXISTE' if ruta.exists() else 'FALTA'}")

# Cambiar al directorio del simulador para que las importaciones relativas funcionen
os.chdir(str(simulador_dir))
# Añadir la ruta absoluta del directorio del simulador al path
sys.path.insert(0, str(simulador_dir))
print(f"Directorio de trabajo cambiado a: {os.getcwd()}")

In [ ]:
# ============================================================================
# PASO 2: Instalar dependencias
# ============================================================================
import subprocess

req_file = Path("requirements.txt")
if not req_file.exists():
    # Buscar requirements.txt en cualquier parte del repo clonado
    reqs = list(Path(".").parent.rglob("requirements.txt"))
    if reqs:
        req_file = reqs[0]
        print(f"requirements.txt encontrado en: {req_file.parent}")
        os.chdir(str(req_file.parent))
    else:
        print("ADVERTENCIA: No se encontró requirements.txt. Instalando dependencias mínimas.")
        req_file = None

if req_file:
    print(f"\nInstalando dependencias desde {req_file} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(req_file)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Fallo al instalar dependencias de requirements.txt")

# scikit-learn es necesario para kernel_cuantico.py pero no está en requirements.txt
print("\nInstalando scikit-learn (necesario para quantum kernel) ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "scikit-learn"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Fallo al instalar scikit-learn")

print("\n✓ Todas las dependencias instaladas")

In [ ]:
# ============================================================================
# PASO 3: Detectar hardware en runtime
# ============================================================================
import sys

from hardware_wheels import detect_gpu_architecture, install_command, quick_summary, print_report

print("=" * 70)
print("DETECCIÓN DE HARDWARE (runtime - no compila nada)")
print("=" * 70)

gpu = detect_gpu_architecture()
hw_summary = quick_summary()

print(f"GPU             : {gpu.name}")
print(f"Arquitectura    : {gpu.architecture}")
print(f"Versión ROCm    : {gpu.rocm_version}")
print(f"Es MI300 (gfx942): {gpu.is_mi300}")
print(f"Es gfx1100      : {gpu.is_gfx1100}")
print(f"Wheel precompilada: {'SÍ' if gpu.supports_precompiled else 'NO'}")
print()
print("Comando de instalación de pennylane-lightning-gpu (bajo demanda):")
print(f"  {install_command(gpu)}")
print()
print("Nota: el simulador PyTorch puro (qsim_torch.py) funciona sin pennylane-lightning-gpu.")
print("Solo instálalo si quieres comparar con lightning.amdgpu.")
print("=" * 70)


In [ ]:
# ============================================================================
# PASO 4: Ejecutar benchmark
# ============================================================================
from benchmark_notebook import run_benchmark

# Configuración para GPU (W7900D o lo que detecte)
resultados = run_benchmark(
    device_str='cuda',        # 'cuda' para GPU AMD/ROCm, 'cpu' para CPU
    ns=[16, 20, 24, 26, 28],  # qubits a probar
    batches=[1, 256, 1024],   # tamaños de lote
    dtypes=None,              # [complex128, complex64] por defecto
    output_dir='.',           # guardar en el directorio actual
)

In [ ]:
# ============================================================================
# PASO 5: Mostrar archivos generados y resumen
# ============================================================================
from pathlib import Path

out = Path(resultados['output_files']['json'])
md = Path(resultados['output_files']['markdown'])

print("=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print(f"JSON (datos brutos):   {out}  ({out.stat().st_size / 1024:.1f} KB)")
print(f"Markdown (resumen):    {md}  ({md.stat().st_size / 1024:.1f} KB)")
print()
print("Contenido del resumen:")
print("=" * 70)
print(md.read_text(encoding='utf-8'))

print("=" * 70)
print("INSTRUCCIONES PARA DESCARGAR")
print("=" * 70)
print("1. Busca los archivos benchmark_results_*.json y BENCHMARK_SUMMARY_*.md")
print("   en el explorador de archivos de Jupyter.")
print("2. Haz clic derecho → Download para bajarlos a tu máquina local.")
print("3. El JSON contiene todos los datos brutos. El Markdown tiene el resumen.")
print()
print("⚠️ IMPORTANTE: Apaga la sesión con 'Turn-off Session' cuando termines.")
print("   La cuota de GPU es 3 h/día y corre aunque la sesión esté inactiva.")

In [ ]:
# Celda de diagnóstico rápido (opcional)
import torch
if torch.cuda.is_available():
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
    print(f"ROCm/HIP: {torch.version.hip}")
else:
    print("No hay GPU disponible. Ejecutando en CPU.")